# Random Forest Calcofi

Queremos determinar cuánta presencia hay y de qué tipo de peces en algunos puntos de la costa Californiana. Particularmente de nuestras variables objetivo:

• Engraulis mordax (Anchoa)

• Sardinops sagax (Sardina)

In [1]:
import pandas as pd

bottle = pd.read_csv("../../data/raw/bottle.csv")
cast = pd.read_csv("../../data/raw/cast.csv")
biologico = pd.read_csv("../../data/raw/biologico/calcofi_ichthyo/occurrence.txt",sep="\t")

# vamos a necesitar el tamaño de los dataset que tenemos

biologico.shape


/tmp/ipykernel_42085/2920896956.py:3: DtypeWarning: Columns (0: IncTim, 1: DIC Quality Comment) have mixed types. Specify dtype option on import or set low_memory=False.
  bottle = pd.read_csv("../../data/raw/bottle.csv")
/tmp/ipykernel_42085/2920896956.py:4: DtypeWarning: Columns (0: Data_Or, 1: Cruz_Num, 2: Inc_Str, 3: Inc_End, 4: PST_LAN, 5: Civil_T) have mixed types. Specify dtype option on import or set low_memory=False.
  cast = pd.read_csv("../../data/raw/cast.csv")


(463655, 13)

Biologico: 463655 Filas con especies de peces, larvas, huevos etc y 13 columnas de observaciones 

In [2]:
cast.shape

(34404, 61)

Cast: 34404 Filas con datos de hora, fecha y lugares de "lanzadas" al mar y 61 columnas de sus condiciones

In [3]:
bottle.shape

(864863, 74)

Bottle: 864863 Filas con sus botellas definidas y 74 columnas de informacion que recolectaron éstas

## 1. Preparación y limpieza de los Datos (Data Preprocessing) 
De éstos 3 dataset, elegimos las variables más relevantes y la unimos en dataset_final


In [4]:
import numpy as np

### Variables Finales del Dataset Unificado (`dataset_final`)

| N° | Columna | Categoría | Descripción / Unidades |
| :---: | :--- | :--- | :--- |
| 1 | `occurrenceID` | Identificador | ID único de la ocurrencia biológica |
| 2 | `scientificName` | Biológica | Nombre científico de la especie (*Sardinops sagax* / *Engraulis mordax*) |
| 3 | `Year` | Temporal | Año del muestreo |
| 4 | `Month` | Temporal | Mes del muestreo |
| 5 | `Lat_Dec` | Espacial | Latitud en grados decimales |
| 6 | `Lon_Dec` | Espacial | Longitud en grados decimales |
| 7 | `Sta_ID` | Estación | Identificador de la estación oceanográfica |
| 8 | `Cruise_ID` | Crucero | Identificador del crucero de investigación |
| 9 | `T_degC` | Físico-Química | Temperatura del agua (°C) |
| 10 | `Salnty` | Físico-Química | Salinidad del agua (PSU) |
| 11 | `STheta` | Físico-Química | Densidad potencial ($\sigma_\theta$) |
| 12 | `O2ml_L` | Físico-Química | Oxígeno disuelto (mL/L) |
| 13 | `O2Sat` | Físico-Química | Saturación de oxígeno (%) |
| 14 | `Oxy_µmol/Kg` | Físico-Química | Oxígeno disuelto ($\mu\text{mol/kg}$) |
| 15 | `ChlorA` | Nutrientes/Pigmentos | Clorofila-a ($\mu\text{g/L}$) |
| 16 | `Phaeop` | Nutrientes/Pigmentos | Feopigmentos ($\mu\text{g/L}$) |
| 17 | `PO4uM` | Nutrientes/Pigmentos | Fosfatos ($\mu\text{mol/L}$) |
| 18 | `SiO3uM` | Nutrientes/Pigmentos | Silicatos ($\mu\text{mol/L}$) |
| 19 | `NO2uM` | Nutrientes/Pigmentos | Nitritos ($\mu\text{mol/L}$) |
| 20 | `NO3uM` | Nutrientes/Pigmentos | Nitratos ($\mu\text{mol/L}$) |
| 21 | `Distance` | Geográfica | Distancia a la costa |
| 22 | `Bottom_D` | Geográfica | Profundidad total del fondo marino ($m$) |
| 23 | `IntChl` | Geográfica | Clorofila integrada en la columna de agua ($mg/m^2$) |
| 24 | `Wind_Spd` | Meteorológica | Velocidad del viento (nudos) |
| 25 | `Wave_Ht` | Meteorológica | Altura de la ola ($m$) |
| 26 | `Barometer` | Meteorológica | Presión barométrica (hPa / mbar) |
| 27 | `Dry_T` | Meteorológica | Temperatura del aire seco (°C) |
| 28 | `Wet_T` | Meteorológica | Temperatura de bulbo húmedo (°C) |


In [5]:
variables_finales = [ "Depthm", "T_degC", "Salnty", "O2ml_L", "STheta", "O2Sat", "Oxy_µmol/Kg", "ChlorA", "Phaeop", "PO4uM", "SiO3uM", "NO2uM", "NO3uM", "NH3uM", "pH1", "pH2", "TA1", "TA2", "DIC1", "DIC2" ]


Nuestra variables objetivos Y: se encuentra en scientificName, Nombre científico de la especie
(*Sardinops sagax* / *Engraulis mordax*)

## Tratamiento de valores nulos 
Es mejor imputarlos (con la mediana o moda). Nosotros vamos a utilizar la mediana así nuestras variables numéricas no se ven afectada por los valores extremos (outliers)

Antes que nada debemos unir algunas variables de Fecha y hora en cada tabla para poder filtrar datos y aplicarles la mediana

In [6]:
# Este paso lo tuve que hacer de nuevo y antes de tratar los valores Nulos porque el EDA es parte de otro documento


# Inspeccionar qué columnas de tiempo/fecha existen en cada tabla
print("--- Columnas en bottle ---")
print([c for c in bottle.columns if any(p in c.lower() for p in ['year', 'date', 'month', 'time'])])

print("\n--- Columnas en cast ---")
print([c for c in cast.columns if any(p in c.lower() for p in ['year', 'date', 'month', 'time'])])

print("\n--- Columnas en biologico ---")
print([c for c in biologico.columns if any(p in c.lower() for p in ['year', 'date', 'month', 'time'])])

--- Columnas en bottle ---
[]

--- Columnas en cast ---
['Date', 'Year', 'Month', 'Julian_Date', 'Time', 'TimeZone']

--- Columnas en biologico ---
[]


Existen algunas en Cast, unimos Bottle y Cast por la variable Cst_Cnt (Recuento de lances: todos los lances de CalCOFI realizados hasta la fecha, numerados consecutivamente)

In [7]:
import pandas as pd

# 1. Unir bottle y cast usando la clave primaria 'Cst_Cnt'
df_ocean = pd.merge(bottle, cast, on='Cst_Cnt', how='inner', suffixes=('', '_cast'))
print(f'df_ocean es bottle + Cast unidos por Cst_Cnt  (muestreos) Filas: {len(df_ocean):,}')

# 2. Ver todas las columnas de 'biologico' para identificar fecha, año y coordenadas
print('\n--- TODAS LAS COLUMNAS DE BIOLOGICO ---')
print(biologico.columns.tolist())

df_ocean es bottle + Cast unidos por Cst_Cnt  (muestreos) Filas: 864,863

--- TODAS LAS COLUMNAS DE BIOLOGICO ---
['id', 'modified', 'basisOfRecord', 'occurrenceID', 'organismQuantity', 'organismQuantityType', 'lifeStage', 'occurrenceStatus', 'preparations', 'eventID', 'scientificNameID', 'scientificName', 'kingdom']


In [8]:
# Uniendo Event que tiene datos de fecha y lugar y biologico que tiene de peces

import pandas as pd

# 1. Cargar el archivo event.txt (contiene fechas, coordenadas y estaciones)
ruta_biolib = "../../data/raw/biologico/calcofi_ichthyo/"
event = pd.read_csv(ruta_biolib + 'event.txt', sep='\t', low_memory=False)

In [9]:
# 2. Unir occurrence con event usando eventID (Identificador del muestreo)
biologico_completo = pd.merge(biologico, event, on='eventID', how='inner')

print(f'Datos biológicos y Event unidos, Filas: {len(biologico_completo):,}')

# 3. Mostrar las nuevas columnas para ver cómo cruzarlas con df_ocean
print('\n--- COLUMNAS DISPONIBLES EN EL BIOLÓGICO COMPLETO ---')
print(biologico_completo.columns.tolist())

Datos biológicos y Event unidos, Filas: 463,655

--- COLUMNAS DISPONIBLES EN EL BIOLÓGICO COMPLETO ---
['id_x', 'modified', 'basisOfRecord', 'occurrenceID', 'organismQuantity', 'organismQuantityType', 'lifeStage', 'occurrenceStatus', 'preparations', 'eventID', 'scientificNameID', 'scientificName', 'kingdom', 'id_y', 'datasetID', 'parentEventID', 'eventType', 'eventDate', 'habitat', 'samplingProtocol', 'sampleSizeValue', 'sampleSizeUnit', 'eventRemarks', 'locationID', 'decimalLatitude', 'decimalLongitude', 'geodeticDatum', 'coordinateUncertaintyInMeters', 'footprintWKT']


Tenemos 463,655 filas biológicas unidas con sus eventos espaciales y temporales correspondientes.

In [10]:
import pandas as pd

# 1. Asegurar que tengamos año y mes listos en biologico_completo a partir de eventDate (Fecha del muestreo biológico)

biologico_completo['eventDate'] = pd.to_datetime(biologico_completo['eventDate'], errors='coerce')
biologico_completo['Year'] = biologico_completo['eventDate'].dt.year
biologico_completo['Month'] = biologico_completo['eventDate'].dt.month


In [11]:

# 2. Redondear coordenadas para facilitar el cruce espacial exacto con las estaciones CalCOFI

biologico_completo['lat_round'] = biologico_completo['decimalLatitude'].round(2)
biologico_completo['lon_round'] = biologico_completo['decimalLongitude'].round(2)
df_ocean['lat_round'] = df_ocean['Lat_Dec'].round(2)
df_ocean['lon_round'] = df_ocean['Lon_Dec'].round(2)

/tmp/ipykernel_42085/1824426403.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ocean['lat_round'] = df_ocean['Lat_Dec'].round(2)
/tmp/ipykernel_42085/1824426403.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ocean['lon_round'] = df_ocean['Lon_Dec'].round(2)


In [12]:

# 1. Filtrar SOLO las dos especies que te interesan antes de fusionar para que no se nos rompa el Kernel, al fin y al cabo solo buscamos 2

especies_objetivo = ['Sardinops sagax', 'Engraulis mordax']
biologico_filtrado = biologico_completo[biologico_completo['scientificName'].isin(especies_objetivo)].copy()

print(f' Registros biológicos para sardina/anchoa: {len(biologico_filtrado):,}')

# 2. Preparar campos de fecha y coordenadas redondeadas en el biológico
biologico_filtrado['eventDate'] = pd.to_datetime(
    biologico_filtrado['eventDate'], errors='coerce'
)
biologico_filtrado['Year'] = biologico_filtrado['eventDate'].dt.year
biologico_filtrado['Month'] = biologico_filtrado['eventDate'].dt.month
biologico_filtrado['lat_round'] = biologico_filtrado['decimalLatitude'].round(2)
biologico_filtrado['lon_round'] = biologico_filtrado['decimalLongitude'].round(2)


 Registros biológicos para sardina/anchoa: 55,370


In [13]:

# Asegurar coordenadas redondeadas en df_ocean (recordemos: df_ocean = bottle y cast unidas por Cst_Cnt (Recuento de lances)
df_ocean['lat_round'] = df_ocean['Lat_Dec'].round(2)
df_ocean['lon_round'] = df_ocean['Lon_Dec'].round(2)

# 3. Hacer el merge final con el subset reducido
dataset_final = pd.merge(df_ocean, biologico_filtrado, on=['Year', 'Month', 'lat_round', 'lon_round'],how='inner',)

print( f'Dimensiones de dataset_final : {dataset_final.shape}')

Dimensiones de dataset_final : (465560, 165)


#### dataset_final es Bottle, Cast, Biologico y Event unidos por fecha , muestreo, lat y long o sea el cuándo, el cómo y dónde. También están los registros de Anchoas y Sardinas

•  Tiene 465560 Filas con 165 columnas con todos sus valores enteros, sin limpiar


# Tratamiento de valores nulos
Ahora veremos que hacer con aquellos valores que son NaN o no dicen nada. Les aplicaremos la mediana para no eliminarlos completamente

In [14]:

from sklearn.impute import SimpleImputer

# 1. Definir las principales variables numéricas / oceanográficas para tu modelo
features_num = ['T_degC', 'Salnty', 'Depthm', 'O2ml_L', 'ChlorA', 'STheta','PO4uM','NO3uM']

# Filtrar solo las columnas que realmente están presentes en tu dataset_final
cols_a_imputar = [c for c in features_num if c in dataset_final.columns]

# 2. Configurar el SimpleImputer utilizando la estrategia 'median'
imputador_mediana = SimpleImputer(strategy='median')

# 3. Aplicar la imputación directamente sobre esas columnas en dataset_final
dataset_final[cols_a_imputar] = imputador_mediana.fit_transform(
    dataset_final[cols_a_imputar]
)

# 4. Comprobar el resultado (debe dar 0 para todas las columnas seleccionadas)
print(' Valores nulos después de imputar con la mediana:')
print(dataset_final[cols_a_imputar].isna().sum())

 Valores nulos después de imputar con la mediana:
T_degC    0
Salnty    0
Depthm    0
O2ml_L    0
ChlorA    0
STheta    0
PO4uM     0
NO3uM     0
dtype: int64


# Conclusión
•  Esto significa que en dataset_final ya no queda ningún valor nulo (NaN) en ellas. Todos los espacios vacíos que había en el dataset fueron rellenados con éxito por la mediana correspondiente de cada variable

# Entrenando el modelo
## División train/test 
Sardina

In [15]:
# 1. Separar los datos específicamente para la variable objetivo Y: Sardina
df_sardina = dataset_final[
    dataset_final['scientificName'] == 'Sardinops sagax'].copy()

In [16]:

# 2. Definir Features (X) y Target (y) para la sardina

features_num = ['T_degC','Salnty','Depthm','O2ml_L','ChlorA','STheta','PO4uM','NO3uM',]
X = df_sardina[features_num]
y = df_sardina['organismQuantity']

print(f'Registros para entrenar modelo de Sardina: {len(df_sardina):,}')

Registros para entrenar modelo de Sardina: 113,656


Anchoa

In [17]:
# 1. Separar los datos específicamente para la variable objetivo Y: Anchoa

df_anchoa = dataset_final[dataset_final['scientificName'] == 'Engraulis mordax'].copy()

: 

In [ ]:
# 2. Definir Features (X) y Target (y) para la anchoa
features_num = ['T_degC','Salnty','Depthm','O2ml_L','ChlorA','STheta','PO4uM','NO3uM',]
X_anchoa = df_anchoa[features_num]
y_anchoa = df_anchoa['organismQuantity']

# convierte los datos de X_anchoa a 32 bits para reducir a la mitad el uso de memoria RAM
X_anchoa = X_anchoa.astype('float32')

print(f'Registros para entrenar modelo de Anchoa: {len(df_anchoa):,}')

Registros para entrenar modelo de Anchoa: 351,904


Dividiendo los datos en conjuntos de entrenamiento y prueba (80% y 20%)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import gc

Sardina

In [ ]:
# 1. Definir Features y Target para la sardina
X_sardina = df_sardina[features_num].astype('float32')
y_sardina = df_sardina['organismQuantity'].astype('float32')

# 2. DIVISIÓN TRAIN/TEST (80% entrenamiento, 20% prueba)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split (X_sardina, y_sardina, test_size=0.2, random_state=42)

gc.collect()

print(f"Tamaño X_train_s: {X_train_s.shape}")
print(f"Tamaño X_test_s: {X_test_s.shape}")

In [ ]:
# 3. Entrenar el modelo Random Forest
print("Entrenando modelo de Sardina...")
rf_sardina = RandomForestRegressor(
    n_estimators=50, 
    max_depth=15, 
    n_jobs=-1, 
    random_state=42)
rf_sardina.fit(X_train_s, y_train_s)

In [ ]:
# 4. Evaluar el rendimiento
y_pred_s = rf_sardina.predict(X_test_s)
r2_s = r2_score(y_test_s, y_pred_s)
rmse_s = np.sqrt(mean_squared_error(y_test_s, y_pred_s))

print("\n Resultados Sardina:")
print(f"  - R²: {r2_s:.4f}")
print(f"  - RMSE: {rmse_s:.4f}")

Anchoa

In [ ]:
# 1. Asegurar features y target para la anchoa
features_num = ['T_degC','Salnty','Depthm','O2ml_L','ChlorA','STheta','PO4uM','NO3uM']
X_anchoa = df_anchoa_sample[features_num].astype('float32')
y_anchoa = df_anchoa_sample['organismQuantity'].astype('float32')

In [ ]:
# 2. DIVISIÓN TRAIN/TEST (80% entrenamiento, 20% prueba)
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split (X_anchoa, y_anchoa, test_size=0.2, random_state=42)

In [ ]:
# Limpiar memoria RAM sobrante
gc.collect()

print(f"Tamaño X_train_a: {X_train_a.shape}")
print(f"Tamaño X_test_a: {X_test_a.shape}")

In [ ]:
# 4. Evaluar el rendimiento
y_pred_a = rf_anchoa.predict(X_test_a)
r2_a = r2_score(y_test_a, y_pred_a)
rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_a))

print("\n Resultados Anchoa:")
print(f"  - R²: {r2_a:.4f}")
print(f"  - RMSE: {rmse_a:.4f}")

In [ ]:
# 3. Entrenar el modelo Random Forest
print("Entrenando modelo de Anchoa...")
rf_anchoa = RandomForestRegressor(
    n_estimators=50, 
    max_depth=15, 
    n_jobs=-1, 
    random_state=42
)
rf_anchoa.fit(X_train_a, y_train_a)

In [ ]:
# 4. Evaluar el rendimiento
y_pred_a = rf_anchoa.predict(X_test_a)
r2_a = r2_score(y_test_a, y_pred_a)
rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_a))

print("\n Resultados Anchoa:")
print(f"  - R²: {r2_a:.4f}")
print(f"  - RMSE: {rmse_a:.4f}")